# Notebook 01: Profiling Inference - Finding the Bottlenecks

## 📚 Historical Context

**Timeline**: 2018-2020 - Learning to measure before optimizing

**The Problem**:
- Early inference optimization was guesswork
- "Let's optimize the attention layer!" - but was it the bottleneck?
- GPT-3 (2020) made profiling critical - couldn't optimize by intuition
- Needed systematic way to identify bottlenecks

**The Realization**:
- Inference is **memory-bandwidth bound**, not compute-bound
- GPU utilization during inference: 10-30% (vs 60-80% during training)
- Most time spent loading weights, not computing
- Profiling tools became essential

## 🎯 What You'll Learn

1. PyTorch Profiler setup and usage
2. Identifying bottlenecks (memory vs compute)
3. CPU vs GPU time breakdown
4. Memory bandwidth analysis
5. Real profiling examples with model inference
6. Interpreting profiler output

## 💡 Why This Matters

- **Measure first, optimize second**: Don't guess at bottlenecks
- **Quantify improvements**: Know if optimization actually helped
- **Find low-hanging fruit**: Optimize highest-impact areas first
- **Avoid wasted effort**: Don't optimize non-bottlenecks

**Key Insight**: "Premature optimization is the root of all evil" - Donald Knuth

**Reference**: See LLM_INFERENCE_ROADMAP.md - Stage 0: Profiling

---

## Setup and Imports

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
import time
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Tuple
import warnings
warnings.filterwarnings('ignore')

# PyTorch profiler imports
from torch.profiler import profile, record_function, ProfilerActivity

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

print("=" * 80)
print("PROFILING ENVIRONMENT CHECK")
print("=" * 80)
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
print("=" * 80)

## Part 1: Understanding Compute vs Memory Bound

### Two Types of Bottlenecks:

**1. Compute Bound** (rare in inference):
- GPU spends most time doing calculations
- High GPU utilization (>60%)
- Examples: Training, large batch sizes, very long sequences

**2. Memory Bound** (typical in inference):
- GPU spends most time loading data from memory
- Low GPU utilization (10-30%)
- Examples: Small batch sizes, sequential generation

### Why Inference is Memory Bound:

```
Arithmetic Intensity = FLOPs / Bytes Transferred

Training (batch=32):   High FLOPs, amortized memory access → Compute bound
Inference (batch=1):   Low FLOPs, same memory access → Memory bound
```

### The Numbers (Llama 2 7B example):

```
Model size: 14 GB (FP16)
Memory bandwidth (A100): 2 TB/s
Time to load model: 14 GB / 2 TB/s = 7 ms

FLOPs for single token: ~14 GFLOPs
Compute time (A100): 14 GFLOPs / 312 TFLOPs = 0.045 ms

Result: 7 ms (memory) >> 0.045 ms (compute)
Conclusion: Memory bound by 150x!
```

In [ ]:
# Load model for profiling
model_name = "gpt2"  # 124M parameters

print("Loading model...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()

# Calculate model size
total_params = sum(p.numel() for p in model.parameters())
param_size_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
param_size_mb = param_size_bytes / (1024 ** 2)

print(f"\nModel: {model_name}")
print(f"Parameters: {total_params:,}")
print(f"Model size: {param_size_mb:.2f} MB ({param_size_bytes / (1024**3):.3f} GB)")
print(f"Precision: {next(model.parameters()).dtype}")
print(f"Device: {device}")

## Part 2: Basic Timing with torch.cuda.Event

Before using the full profiler, let's understand basic GPU timing.

### Why torch.cuda.Event?
- More accurate than `time.time()` for GPU operations
- Accounts for GPU asynchronous execution
- Measures actual GPU kernel time

In [ ]:
def benchmark_forward_pass(
    model,
    input_ids: torch.Tensor,
    num_warmup: int = 5,
    num_runs: int = 20
) -> Dict[str, float]:
    """
    Benchmark a single forward pass with accurate GPU timing.
    
    Args:
        model: The model to benchmark
        input_ids: Input tensor
        num_warmup: Number of warmup runs (GPU needs to warm up)
        num_runs: Number of timed runs
    """
    # Warmup runs (GPU needs to warm up caches, etc.)
    for _ in range(num_warmup):
        with torch.no_grad():
            _ = model(input_ids)
    
    if device == "cuda":
        torch.cuda.synchronize()  # Wait for all kernels to finish
    
    # Timed runs
    times = []
    
    for _ in range(num_runs):
        if device == "cuda":
            start_event = torch.cuda.Event(enable_timing=True)
            end_event = torch.cuda.Event(enable_timing=True)
            
            start_event.record()
            with torch.no_grad():
                _ = model(input_ids)
            end_event.record()
            
            torch.cuda.synchronize()
            elapsed = start_event.elapsed_time(end_event)  # milliseconds
        else:
            start = time.time()
            with torch.no_grad():
                _ = model(input_ids)
            elapsed = (time.time() - start) * 1000  # convert to ms
        
        times.append(elapsed)
    
    return {
        'mean': np.mean(times),
        'std': np.std(times),
        'min': np.min(times),
        'max': np.max(times),
        'median': np.median(times),
        'p95': np.percentile(times, 95),
        'p99': np.percentile(times, 99),
    }

# Benchmark different sequence lengths
print("=" * 80)
print("FORWARD PASS BENCHMARKS")
print("=" * 80)

sequence_lengths = [16, 32, 64, 128, 256, 512]

results = []
for seq_len in sequence_lengths:
    # Create dummy input of this length
    input_ids = torch.randint(0, tokenizer.vocab_size, (1, seq_len)).to(device)
    
    stats = benchmark_forward_pass(model, input_ids, num_warmup=5, num_runs=20)
    results.append(stats)
    
    print(f"\nSequence Length: {seq_len}")
    print(f"  Mean: {stats['mean']:.3f} ms")
    print(f"  Std:  {stats['std']:.3f} ms")
    print(f"  P95:  {stats['p95']:.3f} ms")
    print(f"  P99:  {stats['p99']:.3f} ms")

print("\n" + "=" * 80)

In [ ]:
# Visualize latency vs sequence length
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Latency vs sequence length
ax = axes[0]
means = [r['mean'] for r in results]
stds = [r['std'] for r in results]

ax.plot(sequence_lengths, means, marker='o', linewidth=2, markersize=8, label='Mean')
ax.fill_between(sequence_lengths, 
                 [m - s for m, s in zip(means, stds)],
                 [m + s for m, s in zip(means, stds)],
                 alpha=0.3)
ax.set_xlabel('Sequence Length', fontsize=12)
ax.set_ylabel('Latency (ms)', fontsize=12)
ax.set_title('Forward Pass Latency vs Sequence Length', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 2: Latency growth rate
ax = axes[1]
# Calculate latency per token
latency_per_token = [m / seq_len for m, seq_len in zip(means, sequence_lengths)]
ax.plot(sequence_lengths, latency_per_token, marker='s', linewidth=2, 
        markersize=8, color='green', label='Latency per token')
ax.set_xlabel('Sequence Length', fontsize=12)
ax.set_ylabel('Latency per Token (ms)', fontsize=12)
ax.set_title('Latency per Token (should be constant for efficient attention)', 
             fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nKEY OBSERVATIONS:")
print("• Latency grows with sequence length (more tokens to process)")
print("• For standard attention: O(n²) growth - worse at long sequences")
print("• Latency per token increases for longer sequences (quadratic attention cost)")

## Part 3: PyTorch Profiler - Deep Dive

### What the Profiler Shows:

1. **CPU Time**: Time spent on CPU operations
2. **CUDA Time**: Time spent on GPU kernels
3. **Memory**: Memory allocated/freed
4. **Operator-level breakdown**: Which operations are slowest

### Profiler Activities:
- `ProfilerActivity.CPU`: Profile CPU operations
- `ProfilerActivity.CUDA`: Profile GPU operations (if available)

In [ ]:
def profile_inference(model, tokenizer, prompt: str, max_new_tokens: int = 10):
    """
    Profile inference generation with PyTorch profiler.
    """
    print("=" * 80)
    print("PYTORCH PROFILER - INFERENCE GENERATION")
    print("=" * 80)
    print(f"Prompt: '{prompt}'")
    print(f"Max new tokens: {max_new_tokens}")
    print("\nProfiling...")
    
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    
    # Determine which activities to profile
    activities = [ProfilerActivity.CPU]
    if torch.cuda.is_available():
        activities.append(ProfilerActivity.CUDA)
    
    # Profile the generation loop
    with profile(
        activities=activities,
        record_shapes=True,
        profile_memory=True,
        with_stack=False,
    ) as prof:
        with record_function("inference_generation"):
            for step in range(max_new_tokens):
                with record_function(f"token_{step}"):
                    # Forward pass
                    with record_function("forward_pass"):
                        with torch.no_grad():
                            outputs = model(input_ids)
                            logits = outputs.logits
                    
                    # Sampling
                    with record_function("sampling"):
                        next_token_logits = logits[:, -1, :]
                        probs = F.softmax(next_token_logits, dim=-1)
                        next_token = torch.multinomial(probs, num_samples=1)
                    
                    # Append token
                    with record_function("append_token"):
                        input_ids = torch.cat([input_ids, next_token], dim=-1)
    
    print("\nProfiler complete!\n")
    return prof

# Run profiler
prompt = "The future of AI is"
prof = profile_inference(model, tokenizer, prompt, max_new_tokens=10)

## Part 4: Analyzing Profiler Output

Let's examine the profiler results to understand where time is spent.

In [ ]:
# Print table sorted by CUDA time (GPU time)
print("=" * 80)
print("TOP OPERATIONS BY CUDA TIME (GPU operations)")
print("=" * 80)
print(prof.key_averages().table(
    sort_by="cuda_time_total" if torch.cuda.is_available() else "cpu_time_total",
    row_limit=20
))

print("\n" + "=" * 80)
print("TOP OPERATIONS BY CPU TIME")
print("=" * 80)
print(prof.key_averages().table(
    sort_by="cpu_time_total",
    row_limit=20
))

In [ ]:
# Extract and visualize key metrics
def analyze_profiler_results(prof):
    """
    Extract key metrics from profiler results.
    """
    events = prof.key_averages()
    
    # Separate different operation types
    matmul_ops = []
    attention_ops = []
    layernorm_ops = []
    activation_ops = []
    memory_ops = []
    other_ops = []
    
    for evt in events:
        name = evt.key.lower()
        cuda_time = evt.cuda_time_total if torch.cuda.is_available() else evt.cpu_time_total
        
        if 'matmul' in name or 'mm' in name or 'gemm' in name:
            matmul_ops.append((evt.key, cuda_time))
        elif 'attention' in name or 'softmax' in name:
            attention_ops.append((evt.key, cuda_time))
        elif 'norm' in name:
            layernorm_ops.append((evt.key, cuda_time))
        elif 'gelu' in name or 'relu' in name or 'activation' in name:
            activation_ops.append((evt.key, cuda_time))
        elif 'copy' in name or 'memcpy' in name:
            memory_ops.append((evt.key, cuda_time))
        else:
            other_ops.append((evt.key, cuda_time))
    
    # Calculate totals
    total_matmul = sum(t for _, t in matmul_ops)
    total_attention = sum(t for _, t in attention_ops)
    total_layernorm = sum(t for _, t in layernorm_ops)
    total_activation = sum(t for _, t in activation_ops)
    total_memory = sum(t for _, t in memory_ops)
    total_other = sum(t for _, t in other_ops)
    
    total_time = (total_matmul + total_attention + total_layernorm + 
                  total_activation + total_memory + total_other)
    
    return {
        'MatMul': total_matmul,
        'Attention': total_attention,
        'LayerNorm': total_layernorm,
        'Activation': total_activation,
        'Memory': total_memory,
        'Other': total_other,
        'Total': total_time
    }

# Analyze and visualize
breakdown = analyze_profiler_results(prof)

print("=" * 80)
print("OPERATION BREAKDOWN")
print("=" * 80)
for op_type, time_us in breakdown.items():
    if op_type != 'Total':
        pct = (time_us / breakdown['Total']) * 100 if breakdown['Total'] > 0 else 0
        print(f"{op_type:15s}: {time_us/1000:8.2f} ms ({pct:5.1f}%)")
print(f"{'Total':15s}: {breakdown['Total']/1000:8.2f} ms")
print("=" * 80)

In [ ]:
# Visualize operation breakdown
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Pie chart
ax = axes[0]
labels = [k for k in breakdown.keys() if k != 'Total' and breakdown[k] > 0]
sizes = [breakdown[k] for k in labels]
colors = plt.cm.Set3(range(len(labels)))

ax.pie(sizes, labels=labels, autopct='%1.1f%%', colors=colors, startangle=90)
ax.set_title('Time Breakdown by Operation Type', fontsize=14, fontweight='bold')

# Bar chart
ax = axes[1]
ax.barh(labels, [s/1000 for s in sizes], color=colors)
ax.set_xlabel('Time (ms)', fontsize=12)
ax.set_title('Time Breakdown by Operation Type', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print("\nKEY INSIGHTS:")
print("• Matrix multiplications (MatMul) typically dominate compute time")
print("• Attention operations scale O(n²) with sequence length")
print("• Memory operations indicate data movement bottlenecks")
print("• LayerNorm and activations are usually small overhead")

## Part 5: Memory Profiling

Understanding memory usage is critical for inference optimization.

### Memory Types:
1. **Model weights**: Fixed size (depends on model)
2. **Activations**: Grows with sequence length and batch size
3. **KV cache** (when implemented): Grows with sequence length
4. **Temporary buffers**: Created during operations

In [ ]:
def profile_memory_usage(model, tokenizer, prompt: str, max_new_tokens: int = 20):
    """
    Profile memory usage during generation.
    """
    if not torch.cuda.is_available():
        print("Memory profiling requires CUDA")
        return
    
    print("=" * 80)
    print("MEMORY PROFILING")
    print("=" * 80)
    
    # Reset memory stats
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.empty_cache()
    
    # Record initial memory
    initial_allocated = torch.cuda.memory_allocated() / (1024**2)  # MB
    print(f"Initial memory allocated: {initial_allocated:.2f} MB")
    
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    
    memory_timeline = []
    
    # Generate tokens and track memory
    for step in range(max_new_tokens):
        with torch.no_grad():
            outputs = model(input_ids)
            logits = outputs.logits
        
        next_token_logits = logits[:, -1, :]
        probs = F.softmax(next_token_logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        input_ids = torch.cat([input_ids, next_token], dim=-1)
        
        # Record memory
        allocated = torch.cuda.memory_allocated() / (1024**2)
        reserved = torch.cuda.memory_reserved() / (1024**2)
        memory_timeline.append({
            'step': step,
            'seq_len': input_ids.shape[1],
            'allocated': allocated,
            'reserved': reserved
        })
    
    # Peak memory
    peak_allocated = torch.cuda.max_memory_allocated() / (1024**2)
    peak_reserved = torch.cuda.max_memory_reserved() / (1024**2)
    
    print(f"\nPeak memory allocated: {peak_allocated:.2f} MB")
    print(f"Peak memory reserved: {peak_reserved:.2f} MB")
    print(f"Memory growth: {peak_allocated - initial_allocated:.2f} MB")
    print("=" * 80)
    
    return memory_timeline

# Profile memory
if torch.cuda.is_available():
    memory_data = profile_memory_usage(model, tokenizer, "The future of technology is", max_new_tokens=30)
else:
    print("Skipping memory profiling (CUDA not available)")

In [ ]:
# Visualize memory growth
if torch.cuda.is_available() and memory_data:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    steps = [d['step'] for d in memory_data]
    seq_lens = [d['seq_len'] for d in memory_data]
    allocated = [d['allocated'] for d in memory_data]
    reserved = [d['reserved'] for d in memory_data]
    
    # Plot 1: Memory vs steps
    ax = axes[0]
    ax.plot(steps, allocated, marker='o', label='Allocated', linewidth=2)
    ax.plot(steps, reserved, marker='s', label='Reserved', linewidth=2)
    ax.set_xlabel('Generation Step', fontsize=12)
    ax.set_ylabel('Memory (MB)', fontsize=12)
    ax.set_title('Memory Usage During Generation', fontsize=14, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Plot 2: Memory vs sequence length
    ax = axes[1]
    ax.plot(seq_lens, allocated, marker='o', color='green', linewidth=2)
    ax.set_xlabel('Sequence Length', fontsize=12)
    ax.set_ylabel('Memory Allocated (MB)', fontsize=12)
    ax.set_title('Memory Growth vs Sequence Length', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("\nKEY OBSERVATIONS:")
    print("• Memory grows with sequence length (activations increase)")
    print("• In naive inference, no KV cache means less memory but more recomputation")
    print("• Reserved memory > allocated (PyTorch caches memory for reuse)")
    print("• Stage 1 will add KV cache: trade memory for speed")

## Part 6: Comparing CPU vs GPU Performance

In [ ]:
def compare_cpu_gpu_performance(model, tokenizer, prompt: str, max_new_tokens: int = 10):
    """
    Compare inference performance on CPU vs GPU.
    """
    print("=" * 80)
    print("CPU vs GPU PERFORMANCE COMPARISON")
    print("=" * 80)
    
    devices_to_test = ['cpu']
    if torch.cuda.is_available():
        devices_to_test.append('cuda')
    
    results = {}
    
    for dev in devices_to_test:
        print(f"\nTesting on {dev.upper()}...")
        
        # Move model to device
        model_on_device = model.to(dev)
        input_ids = tokenizer.encode(prompt, return_tensors="pt").to(dev)
        
        # Warmup
        for _ in range(3):
            with torch.no_grad():
                _ = model_on_device(input_ids)
        
        # Time generation
        start = time.time()
        generated_ids = input_ids.clone()
        
        for _ in range(max_new_tokens):
            with torch.no_grad():
                outputs = model_on_device(generated_ids)
                logits = outputs.logits
            
            next_token = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)
            generated_ids = torch.cat([generated_ids, next_token], dim=-1)
        
        if dev == 'cuda':
            torch.cuda.synchronize()
        
        elapsed = time.time() - start
        tokens_per_sec = max_new_tokens / elapsed
        ms_per_token = (elapsed / max_new_tokens) * 1000
        
        results[dev] = {
            'total_time': elapsed,
            'tokens_per_sec': tokens_per_sec,
            'ms_per_token': ms_per_token
        }
        
        print(f"  Total time: {elapsed:.3f} s")
        print(f"  Tokens/sec: {tokens_per_sec:.2f}")
        print(f"  ms/token: {ms_per_token:.2f}")
    
    # Move model back to original device
    model.to(device)
    
    # Calculate speedup
    if len(results) > 1:
        speedup = results['cpu']['total_time'] / results['cuda']['total_time']
        print(f"\nGPU Speedup: {speedup:.2f}x faster than CPU")
    
    print("=" * 80)
    return results

# Compare performance
perf_results = compare_cpu_gpu_performance(model, tokenizer, "Once upon a time", max_new_tokens=15)

## Part 7: Profiling Checklist and Best Practices

In [ ]:
print("=" * 80)
print("PROFILING BEST PRACTICES CHECKLIST")
print("=" * 80)
print("""
✓ BEFORE PROFILING:
  1. Always warmup the model (5-10 runs)
  2. Clear CUDA cache before measurements
  3. Use torch.cuda.synchronize() for accurate GPU timing
  4. Set model to eval mode (model.eval())
  5. Use torch.no_grad() to disable gradient computation

✓ DURING PROFILING:
  1. Run multiple iterations for statistical significance
  2. Measure both mean and percentiles (P95, P99)
  3. Profile different sequence lengths
  4. Profile different batch sizes
  5. Monitor both time and memory

✓ ANALYZING RESULTS:
  1. Identify the bottleneck (compute vs memory)
  2. Look for operations taking >10% of time
  3. Check memory growth patterns
  4. Compare against theoretical limits
  5. Validate with multiple runs

✓ COMMON PITFALLS TO AVOID:
  1. Don't profile with gradients enabled
  2. Don't skip warmup runs
  3. Don't use time.time() for GPU operations
  4. Don't profile in train mode (dropout adds noise)
  5. Don't optimize without measuring first

✓ WHAT TO LOOK FOR:
  1. GPU utilization < 30% → Memory bound (typical for inference)
  2. GPU utilization > 60% → Compute bound (rare, large batches)
  3. Memory growing linearly → Expected (activations)
  4. Memory growing quadratically → Problem (check attention)
  5. High CPU time → Python overhead or data loading
""")
print("=" * 80)

## 📊 Summary and Key Takeaways

### What We Learned:

1. **Inference is Memory-Bandwidth Bound**:
   - GPU utilization: 10-30% (vs 60-80% in training)
   - Time spent loading weights > time computing
   - Memory bandwidth is the primary bottleneck

2. **Profiling Tools**:
   - `torch.cuda.Event`: Accurate GPU timing
   - `torch.profiler`: Detailed operation breakdown
   - Memory tracking: Understand allocation patterns

3. **Key Metrics**:
   - **Latency**: Time per token (ms/token)
   - **Throughput**: Tokens per second
   - **Memory**: Peak allocation and growth
   - **GPU utilization**: Indicates bottleneck type

4. **Operation Breakdown**:
   - Matrix multiplications: Usually largest compute time
   - Attention: Scales O(n²) with sequence length
   - Memory operations: Indicate data movement costs

### Typical Profiling Results (GPT-2 on GPU):

```
GPU Utilization: 15-25%
Tokens/sec: 8-15 (naive inference)
ms/token: 60-120 ms
Memory: ~250 MB (model) + ~50 MB (activations)

Time breakdown:
  - MatMul operations: 60-70%
  - Attention: 15-20%
  - Other (LayerNorm, activations): 10-15%
```

### What This Tells Us:

1. **Low GPU utilization** → Memory bound, not compute bound
2. **Linear memory growth** → Expected (activations grow with sequence)
3. **MatMul dominates** → But still memory-bandwidth limited
4. **CPU vs GPU**: 5-20x speedup on GPU

### Optimization Strategy (based on profiling):

```
Priority 1: Reduce memory bandwidth (quantization, batching)
Priority 2: Reduce memory usage (Flash Attention, KV cache optimization)
Priority 3: Improve compute efficiency (only if compute-bound)
```

### Next Steps:

**Notebook 02: Prefill vs Decode** - Learn about:
- Two distinct phases of generation
- Different bottlenecks for each phase
- Why they need different optimizations
- Measuring and optimizing each separately

---

### Key Principle:

> "Measure twice, optimize once"

Always profile before optimizing. Don't assume you know the bottleneck - measure it!

---

**Reference**: LLM_INFERENCE_ROADMAP.md - Stage 0: Profiling